In [0]:
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable

login = "mariiasirska"
catalog = "dbr_dev_ua5816bd"
bronze = f"{catalog}.{login}_bronze"
silver = f"{catalog}.{login}_silver"

def bt(name):
    return f"{bronze}.olist_{name}_bronze"

def st(name):
    return f"{silver}.olist_{name}_silver"

QUAR = f"{silver}.olist_quarantine"
LOG = f"{silver}.olist_pipeline_log"

In [0]:
META = "_ingested_at TIMESTAMP, _source_file STRING, _batch_id INT, _silver_updated_at TIMESTAMP"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {st('orders')} (
  order_id STRING NOT NULL,
  customer_id STRING NOT NULL,
  order_status STRING,
  purchase_ts TIMESTAMP,
  approved_ts TIMESTAMP,
  delivered_carrier_ts TIMESTAMP,
  delivered_customer_ts TIMESTAMP,
  estimated_delivery_ts TIMESTAMP,
  {META}
)""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {st('customers')} (
  customer_id STRING NOT NULL,
  customer_unique_id STRING,
  zip_code_prefix STRING,
  city STRING,
  state STRING,
  {META}
)""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {st('products')} (
  product_id STRING NOT NULL,
  category_name STRING,
  category_name_en STRING,
  name_length INT,
  description_length INT,
  photos_qty INT,
  weight_g INT,
  length_cm INT,
  height_cm INT,
  width_cm INT,
  {META}
)""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {st('order_items')} (
  order_id STRING NOT NULL,
  order_item_id INT NOT NULL,
  product_id STRING,
  seller_id STRING,
  shipping_limit_ts TIMESTAMP,
  price DECIMAL(10,2),
  freight_value DECIMAL(10,2),
  {META}
)""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {st('reviews')} (
  review_id STRING NOT NULL,
  order_id STRING NOT NULL,
  review_score INT,
  review_comment_title STRING,
  review_comment_message STRING,
  review_creation_ts TIMESTAMP,
  review_answer_ts TIMESTAMP,
  {META}
)""")


SCD Type 2 dimension:

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {st('sellers')} (
  seller_id STRING NOT NULL,
  zip_code_prefix STRING,
  city STRING,
  state STRING,
  effective_from TIMESTAMP,
  effective_to TIMESTAMP,
  is_current BOOLEAN,
  {META}
)""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {QUAR} (
  table_name STRING, reason STRING, payload STRING, quarantined_at TIMESTAMP
)""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {LOG} (
  table_name STRING, batch_id INT, rows_in BIGINT, rows_valid BIGINT,
  rows_quarantined BIGINT, processed_at TIMESTAMP
)""")


In [0]:
def add_meta(df):
    return (
        df.withColumn("_ingested_at", F.col("ingestion_ts"))
        .withColumn("_source_file", F.col("source_file"))
        .withColumn("_batch_id", F.col("batch_id").cast("int"))
        .withColumn("_silver_updated_at", F.current_timestamp())
    )

In [0]:
def align(df, name):
    return df.select(*spark.table(st(name)).columns)

def latest(df, keys):
    w = Window.partitionBy(*keys).orderBy(F.col("_ingested_at").desc(), F.col("_batch_id").desc())
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

In [0]:
def split_valid(df, rules):
    ok = F.lit(True)
    for cond in rules.values():
        ok = ok & F.coalesce(cond, F.lit(False))
    reason = F.concat_ws(
        "; ", *[F.when(~F.coalesce(c, F.lit(False)), F.lit(r)) for r, c in rules.items()]
    )
    flagged = df.withColumn("_dq_ok", ok).withColumn("_dq_reason", reason)
    valid = flagged.filter("_dq_ok").drop("_dq_ok", "_dq_reason")
    bad = flagged.filter("NOT _dq_ok")
    return valid, bad

def to_quarantine(bad, name):
    payload_cols = [c for c in bad.columns if not c.startswith("_dq")]
    return bad.select(
        F.lit(name).alias("table_name"),
        F.col("_dq_reason").alias("reason"),
        F.to_json(F.struct(*payload_cols)).alias("payload"),
        F.current_timestamp().alias("quarantined_at"),
    )

def pending_batches(name):
    done = spark.table(LOG).filter(F.col("table_name") == name).select("batch_id")
    rows = (
        spark.table(bt(name)).select("batch_id").distinct()
        .join(done, "batch_id", "left_anti").orderBy("batch_id").collect()
    )
    return [int(r["batch_id"]) for r in rows]

In [0]:
def scd1(name, keys):
    def fn(src):
        cond = " AND ".join(f"t.{k} = s.{k}" for k in keys)
        (
            DeltaTable.forName(spark, st(name)).alias("t")
            .merge(src.alias("s"), cond)
            .whenMatchedUpdateAll(condition="s._ingested_at > t._ingested_at")
            .whenNotMatchedInsertAll()
            .execute()
        )
    return fn

In [0]:
def process(name, transform, keys, rules, merge_fn, bronze_name=None):
    bronze_name = bronze_name or name
    pending = pending_batches(bronze_name)
    if not pending:
        print(f"{name}: nothing to process")
        return
    for b in pending:
        src = spark.table(bt(bronze_name)).filter(F.col("batch_id") == b)
        rows_in = src.count()
        valid, bad = split_valid(transform(src), rules)
        valid = latest(valid, keys)
        rows_valid = valid.count()
        merge_fn(valid)

        bad_df = to_quarantine(bad, name)
        n_bad = bad_df.count()
        if n_bad:
            bad_df.write.mode("append").saveAsTable(QUAR)

        (
            spark.createDataFrame(
                [(bronze_name, b, rows_in, rows_valid, n_bad)],
                "table_name string, batch_id int, rows_in long, rows_valid long, rows_quarantined long",
            )
            .withColumn("processed_at", F.current_timestamp())
            .write.mode("append").saveAsTable(LOG)
        )
        print(f"{name} batch {b}: in={rows_in}, valid(deduped)={rows_valid}, quarantined={n_bad}")

Transformations and quality rules:

In [0]:
STATUSES = ["created", "approved", "invoiced", "processing", "shipped", "delivered", "unavailable", "canceled"]

In [0]:
def t_orders(df):
    df = (
        add_meta(df)
        .withColumn("order_status", F.lower(F.trim("order_status")))
        .withColumn("purchase_ts", F.to_timestamp("order_purchase_timestamp"))
        .withColumn("approved_ts", F.to_timestamp("order_approved_at"))
        .withColumn("delivered_carrier_ts", F.to_timestamp("order_delivered_carrier_date"))
        .withColumn("delivered_customer_ts", F.to_timestamp("order_delivered_customer_date"))
        .withColumn("estimated_delivery_ts", F.to_timestamp("order_estimated_delivery_date"))
    )
    return align(df, "orders")

In [0]:
def t_customers(df):
    df = (
        add_meta(df)
        .withColumn("zip_code_prefix", F.col("customer_zip_code_prefix"))
        .withColumn("city", F.lower(F.trim("customer_city")))
        .withColumn("state", F.upper(F.trim("customer_state")))
    )
    return align(df, "customers")

In [0]:
def t_products(df):
    tr = (
        spark.table(bt("category_translation"))
        .select("product_category_name", F.col("product_category_name_english").alias("category_name_en"))
        .dropDuplicates(["product_category_name"])
    )
    df = (
        add_meta(df)
        .join(F.broadcast(tr), "product_category_name", "left")
        .withColumn("category_name", F.col("product_category_name"))
        .withColumn("name_length", F.col("product_name_lenght").cast("int"))
        .withColumn("description_length", F.col("product_description_lenght").cast("int"))
        .withColumn("photos_qty", F.col("product_photos_qty").cast("int"))
        .withColumn("weight_g", F.col("product_weight_g").cast("int"))
        .withColumn("length_cm", F.col("product_length_cm").cast("int"))
        .withColumn("height_cm", F.col("product_height_cm").cast("int"))
        .withColumn("width_cm", F.col("product_width_cm").cast("int"))
    )
    return align(df, "products")

In [0]:
def t_items(df):
    df = (
        add_meta(df)
        .withColumn("order_item_id", F.col("order_item_id").cast("int"))
        .withColumn("shipping_limit_ts", F.to_timestamp("shipping_limit_date"))
        .withColumn("price", F.col("price").cast("decimal(10,2)"))
        .withColumn("freight_value", F.col("freight_value").cast("decimal(10,2)"))
    )
    return align(df, "order_items")

In [0]:
def t_reviews(df):
    df = (
        add_meta(df)
        .withColumn("review_score", F.col("review_score").cast("int"))
        .withColumn("review_creation_ts", F.to_timestamp("review_creation_date"))
        .withColumn("review_answer_ts", F.to_timestamp("review_answer_timestamp"))
    )
    return align(df, "reviews")

In [0]:
def t_sellers(df):
    df = (
        add_meta(df)
        .withColumn("zip_code_prefix", F.col("seller_zip_code_prefix"))
        .withColumn("city", F.lower(F.trim("seller_city")))
        .withColumn("state", F.upper(F.trim("seller_state")))
        .withColumn("effective_from", F.col("_ingested_at"))
        .withColumn("effective_to", F.lit(None).cast("timestamp"))
        .withColumn("is_current", F.lit(True))
    )
    return align(df, "sellers")

rules = {
    "orders": {
        "order_id is null": F.col("order_id").isNotNull(),
        "customer_id is null": F.col("customer_id").isNotNull(),
        "invalid order_status": F.col("order_status").isin(STATUSES),
    },
    "customers": {"customer_id is null": F.col("customer_id").isNotNull()},
    "products": {
        "product_id is null": F.col("product_id").isNotNull(),
        "negative weight": F.col("weight_g").isNull() | (F.col("weight_g") >= 0),
    },
    "order_items": {
        "order_id is null": F.col("order_id").isNotNull(),
        "order_item_id is null": F.col("order_item_id").isNotNull(),
        "negative price": F.col("price") >= 0,
    },
    "reviews": {
        "review_id is null": F.col("review_id").isNotNull(),
        "order_id is null": F.col("order_id").isNotNull(),
        "review_score out of 1..5": F.col("review_score").between(1, 5),
    },
    "sellers": {"seller_id is null": F.col("seller_id").isNotNull()},
}


SCD Type 2 MERGE for sellers:

In [0]:
def scd2_sellers(src):
    tgt_name = st("sellers")
    cols = spark.table(tgt_name).columns
    current = spark.table(tgt_name).filter("is_current")
    same = "t.city <=> s.city AND t.state <=> s.state AND t.zip_code_prefix <=> s.zip_code_prefix"

    new_versions = (
        src.alias("s")
        .join(current.alias("t"), F.col("s.seller_id") == F.col("t.seller_id"))
        .filter(f"NOT ({same})")
        .select("s.*")
    )
    staged = src.withColumn("merge_key", F.col("seller_id")).unionByName(
        new_versions.withColumn("merge_key", F.lit(None).cast("string"))
    )

    (
        DeltaTable.forName(spark, tgt_name).alias("t")
        .merge(staged.alias("s"), "t.seller_id = s.merge_key AND t.is_current = true")
        .whenMatchedUpdate(
            condition=f"NOT ({same})",
            set={"is_current": "false", "effective_to": "s.effective_from"},
        )
        .whenNotMatchedInsert(values={c: f"s.{c}" for c in cols})
        .execute()
    )

In [0]:
def run_all():
    process("orders", t_orders, ["order_id"], rules["orders"], scd1("orders", ["order_id"]))
    process("customers", t_customers, ["customer_id"], rules["customers"], scd1("customers", ["customer_id"]))
    process("products", t_products, ["product_id"], rules["products"], scd1("products", ["product_id"]))
    process("order_items", t_items, ["order_id", "order_item_id"], rules["order_items"],
            scd1("order_items", ["order_id", "order_item_id"]))
    process("reviews", t_reviews, ["review_id", "order_id"], rules["reviews"],
            scd1("reviews", ["review_id", "order_id"]), bronze_name="order_reviews")
    process("sellers", t_sellers, ["seller_id"], rules["sellers"], scd2_sellers)

In [0]:
run_all()

In [0]:
display(spark.table(LOG).orderBy("table_name", "batch_id"))

SCD1: order status overwritten, MERGE metrics per version:

In [0]:
display(
    spark.sql(f"DESCRIBE HISTORY {st('orders')}")
    .filter("operation = 'MERGE'")
    .selectExpr(
        "version", "timestamp",
        "operationMetrics['numTargetRowsInserted'] as inserted",
        "operationMetrics['numTargetRowsUpdated'] as updated",
    )
    .orderBy("version")
)

SCD2: sellers with more than one version:

In [0]:
display(
    spark.sql(f"""
    SELECT seller_id, city, state, effective_from, effective_to, is_current
    FROM {st('sellers')}
    WHERE seller_id IN (SELECT seller_id FROM {st('sellers')} GROUP BY seller_id HAVING count(*) > 1)
    ORDER BY seller_id, effective_from
    """).limit(20)
)

Deduplication: raw reviews vs silver reviews:

In [0]:
print("bronze reviews rows:", spark.table(bt("order_reviews")).count())
print("silver reviews rows:", spark.table(st("reviews")).count())
display(spark.table(QUAR).groupBy("table_name", "reason").count())

Idempotent re-run:

In [0]:
before = {t: spark.table(st(t)).count() for t in ["orders", "customers", "products", "order_items", "reviews", "sellers"]}
run_all()
after = {t: spark.table(st(t)).count() for t in before}
print("counts unchanged after re-run:", before == after)
print(after)
